In [1]:
# Cell 0: Session Restore
# Run this cell first every session
# Loads the final modeling dataset produced by Notebook 2

from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Paths
drive_root = '/content/drive/MyDrive/mets-risk-score/'
data_raw   = os.path.join(drive_root, 'data/raw/')
data_proc  = os.path.join(drive_root, 'data/processed/')
fig_dir    = os.path.join(drive_root, 'outputs/figures/')
src_dir    = os.path.join(drive_root, 'src/')

# Load modeling dataset
X = pd.read_csv(os.path.join(data_proc, 'x_modeling.csv'), low_memory=False)
y = pd.read_csv(os.path.join(data_proc, 'y_msss.csv')).squeeze()

# Load modeling dataset
X = pd.read_csv(os.path.join(data_proc, 'x_modeling.csv'), low_memory=False)
y = pd.read_csv(os.path.join(data_proc, 'y_msss.csv')).squeeze()

print('Session restored\n')
print(f'X (predictors) : {X.shape[0]:,} rows × {X.shape[1]:,} columns')
print(f'y (MSSS target): {y.shape[0]:,} rows')
print(f'y mean         : {y.mean():.3f}')
print(f'y std          : {y.std():.3f}')


Mounted at /content/drive
Session restored

X (predictors) : 1,203 rows × 435 columns
y (MSSS target): 1,203 rows
y mean         : 0.010
y std          : 1.104


In [2]:
# Cell 1: Column-level Missingness check
# Before imputation we need to understand the missingness structure:
# This audit drives every downstream decision about what goes into the MICE imputer
#  and what gets dropped before modeling.

# Missingness per column
missing_count = X.isnull().sum()
missing_pct = (missing_count / len(X) * 100).round(2)

missingness_df = pd.DataFrame({
    'column' : missing_count.index,
    'missing_n' : missing_count.values,
    'missing_pct' : missing_pct.values,
}).sort_values('missing_pct', ascending=False).reset_index(drop = True)

# Bucket columns by missingness level
complete = missingness_df[missingness_df['missing_pct'] == 0]
low = missingness_df[(missingness_df['missing_pct'] > 0) &
                     (missingness_df['missing_pct'] <= 20)]
moderate = missingness_df[(missingness_df['missing_pct'] >  20) &
                              (missingness_df['missing_pct'] <= 40)]
high = missingness_df[(missingness_df['missing_pct'] >  40) &
                              (missingness_df['missing_pct'] <  100)]
all_missing = missingness_df[missingness_df['missing_pct'] == 100]

print(f"Total columns :{X.shape[1]}")
print(f"Total Rows : {X.shape[0]:,}\n")

print(f"Complete (0% missing) :{len(complete): > 4} columns")
print(f"low (1-20% missing) :{len(low): > 4} columns")
print(f"Moderate (21-40%% missing) :{len(moderate): > 4} columns")
print(f"Hig (41-99%% missing) :{len(high): > 4} columns")
print(f"All missing :{len(all_missing): > 4} columns")

Total columns :435
Total Rows : 1,203

Complete (0% missing) :  54 columns
low (1-20% missing) : 255 columns
Moderate (21-40%% missing) :   6 columns
Hig (41-99%% missing) : 108 columns
All missing :  12 columns


In [ ]:
# CELL 2: Structural Zero Fill + Feature Engineering + Missingness Drop

# Step 1: copy
X_clean = X.copy()
print(f"{X_clean.shape[0]:,} rows × {X_clean.shape[1]:,} columns")

# Step 2: Fill structural zeros for alcohol follow-ups
# Non-drinkers identified from base questions:
#   ALQ111 == 2  never had a drink in their life
#   ALQ121 == 0  never drank in past 12 months
non_drinker_mask = (X_clean['ALQ111'] == 2) | (X_clean['ALQ121'] == 0)

alcohol_followup_cols = [
    'ALQ130',  # avg drinks per day when drinking
    'ALQ142',  # days had 5+ drinks in past 12 months
    'ALQ170',  # binge drinking past 30 days
    'ALQ270',  # binge drinking frequency
    'ALQ280',  # days drunk or very high past year
    'ALQ290',  # days felt sick from drinking past year
]

# Only include columns that exist
alcohol_followup_cols = [c for c in alcohol_followup_cols
                         if c in X_clean.columns]

before_fill = X_clean[alcohol_followup_cols].isnull().sum()
X_clean.loc[non_drinker_mask, alcohol_followup_cols] = \
    X_clean.loc[non_drinker_mask, alcohol_followup_cols].fillna(0)
after_fill = X_clean[alcohol_followup_cols].isnull().sum()

print(f"\n Alcohol structural zero fill:")
print(f"   Non-drinkers identified : {non_drinker_mask.sum():,}")
for col in alcohol_followup_cols:
    filled = before_fill[col] - after_fill[col]
    print(f"   {col:<12} {before_fill[col]:>8,} "
          f"{after_fill[col]:>8,} {filled:>8,}")

# Step 3: Engineer total_MET_minutes
# Published NHANES formula for total physical activity (MET-min/week).
# Source: Global Physical Activity Questionnaire (GPAQ) methodology,
#         confirmed from NHANES PAQ documentation and published research.
#
# Formula:
#   total_MET = (days_vigorous_work   × min_vigorous_work   × 8) +
#               (days_moderate_work   × min_moderate_work   × 4) +
#               (days_walking         × min_walking         × 4) +
#               (days_vigorous_rec    × min_vigorous_rec    × 8) +
#               (days_moderate_rec    × min_moderate_rec    × 4)
#
# MET multipliers:
#   8 = vigorous activity (doubles moderate MET score)
#   4 = moderate activity
#
# Inactive people naturally get 0 - no imputation needed.
# NaN * anything = NaN, so we fill PAQ base questions with 0
# where participant indicated no activity (already confirmed complete).


# Fill PAD duration columns with 0 where base question = no activity
# Base question coding: 1 = yes, 2 = no
pad_fill_map = {
    'PAD615': ('PAQ605', 2),  # vigorous work duration - fill if no vig work
    'PAD630': ('PAQ620', 2),  # moderate work duration - fill if no mod work
    'PAD645': ('PAQ635', 2),  # walking duration       - fill if no walking
    'PAD660': ('PAQ650', 2),  # vigorous rec duration  - fill if no vig rec
    'PAD675': ('PAQ665', 2),  # moderate rec duration  - fill if no mod rec
}

for pad_col, (paq_col, no_code) in pad_fill_map.items():
    if pad_col in X_clean.columns and paq_col in X_clean.columns:
        inactive_mask = X_clean[paq_col] == no_code
        filled = X_clean.loc[inactive_mask, pad_col].isnull().sum()
        X_clean.loc[inactive_mask, pad_col] = \
            X_clean.loc[inactive_mask, pad_col].fillna(0)
        print(f"   {pad_col} ← filled {filled:,} zeros "
              f"(inactive per {paq_col})")

# Compute MET-minutes per week
# Days columns (PAQ610, PAQ625, PAQ640, PAQ655, PAQ670) * duration * MET
met_components = {
    'vigorous_work' :    ('PAQ610', 'PAD615', 8),
    'moderate_work' :    ('PAQ625', 'PAD630', 4),
    'walking'       :    ('PAQ640', 'PAD645', 4),
    'vigorous_rec'  :    ('PAQ655', 'PAD660', 8),
    'moderate_rec'  :    ('PAQ670', 'PAD675', 4),
}

X_clean['total_MET_minutes'] = 0.0

for component, (days_col, min_col, met) in met_components.items():
    if days_col in X_clean.columns and min_col in X_clean.columns:
        contribution = X_clean[days_col].fillna(0) * \
                       X_clean[min_col].fillna(0) * met
        X_clean['total_MET_minutes'] += contribution

print(f"\n   total_MET_minutes distribution:")
print(f"   Mean   : {X_clean['total_MET_minutes'].mean():.1f}")
print(f"   Median : {X_clean['total_MET_minutes'].median():.1f}")
print(f"   Min    : {X_clean['total_MET_minutes'].min():.1f}")
print(f"   Max    : {X_clean['total_MET_minutes'].max():.1f}")
print(f"   Zero   : {(X_clean['total_MET_minutes'] == 0).sum():,} "
      f"participants (completely inactive)")

# Drop individual PAQ/PAD columns now replaced by total_MET_minutes
# Keep PAD680 (sedentary minutes) — separate meaningful predictor
paq_pad_to_drop = [
    'PAQ605', 'PAQ610', 'PAQ620', 'PAQ625',
    'PAQ635', 'PAQ640', 'PAQ650', 'PAQ655',
    'PAQ665', 'PAQ670',
    'PAD615', 'PAD630', 'PAD645', 'PAD660', 'PAD675',
]
paq_pad_to_drop = [c for c in paq_pad_to_drop if c in X_clean.columns]
X_clean = X_clean.drop(columns=paq_pad_to_drop)

# Log-transform total_MET_minutes
# total_MET_minutes is right-skewed (max 125,748, 24% above WHO threshold).
# Log(1 + x) transformation compresses extreme values without dropping
# anyone. The +1 handles zero-activity participants (log(1+0) = 0).
# Standard practice in published NHANES physical activity research.

X_clean['total_MET_minutes'] = np.log1p(X_clean['total_MET_minutes'])

print(f"\n   After log(1+x) transform:")
print(f"   Mean   : {X_clean['total_MET_minutes'].mean():.3f}")
print(f"   Median : {X_clean['total_MET_minutes'].median():.3f}")
print(f"   Min    : {X_clean['total_MET_minutes'].min():.3f}")
print(f"   Max    : {X_clean['total_MET_minutes'].max():.3f}")

print(f"\n   PAQ/PAD columns dropped    : {len(paq_pad_to_drop)}")
print(f"   Replaced by                : log_total_MET_minutes + PAD680 (sedentary)")

#Step 4: Recalculate TRUE missingness
missing_count = X_clean.isnull().sum()
missing_pct   = (missing_count / len(X_clean) * 100).round(2)

missingness_df = pd.DataFrame({
    'column'      : missing_count.index,
    'missing_n'   : missing_count.values,
    'missing_pct' : missing_pct.values,
}).sort_values('missing_pct', ascending=False).reset_index(drop=True)

complete    = missingness_df[missingness_df['missing_pct'] == 0]
low         = missingness_df[(missingness_df['missing_pct'] >  0) &
                              (missingness_df['missing_pct'] <= 20)]
moderate    = missingness_df[(missingness_df['missing_pct'] >  20) &
                              (missingness_df['missing_pct'] <= 40)]
high        = missingness_df[(missingness_df['missing_pct'] >  40) &
                              (missingness_df['missing_pct'] <  100)]
all_missing = missingness_df[missingness_df['missing_pct'] == 100]

print(f"\n True Missingness After Zero-Fill & Engineering:")
print(f"    Complete  (0%)        : {len(complete):>4} columns")
print(f"    Low       (1–20%)     : {len(low):>4} columns - MICE")
print(f"    Moderate  (21–40%)    : {len(moderate):>4} columns - MICE")
print(f"    High      (41–99%)    : {len(high):>4} columns - drop")
print(f"    All missing (100%)    : {len(all_missing):>4} columns - drop")

# Step 5: Drop unusable columns
drop_log = {}

# Drop 1: 100% missing
cols_all_missing = all_missing['column'].tolist()
X_clean = X_clean.drop(columns=cols_all_missing, errors='ignore')
drop_log['100% missing'] = cols_all_missing

# Drop 2: >40% true missing
cols_high_missing = high['column'].tolist()
X_clean = X_clean.drop(columns=cols_high_missing, errors='ignore')
drop_log['>40% true missing'] = cols_high_missing

# Drop 3: Near-zero variance (>95% same value)
nzv_cols = []
for col in X_clean.select_dtypes(include=[np.number]).columns:
    top_freq = X_clean[col].value_counts(normalize=True).iloc[0]
    if top_freq >= 0.95:
        nzv_cols.append(col)
X_clean = X_clean.drop(columns=nzv_cols, errors='ignore')
drop_log['near-zero variance'] = nzv_cols

# Step 6: Drop summary
print(f"\n Drop Summary:")
total_dropped = 0
for reason, cols in drop_log.items():
    print(f"   {reason:<25} : {len(cols):>4} columns dropped")
    total_dropped += len(cols)

print(f"\n   Total columns dropped  : {total_dropped}")
print(f"   Columns remaining      : {X_clean.shape[1]}")
print(f"   Rows unchanged         : {X_clean.shape[0]:,}")

# Show high-missing columns dropped for transparency
if cols_high_missing:
    print(f"\n   Columns dropped (>40% true missing):")
    for col in cols_high_missing[:20]:  # show top 20
        pct = missingness_df[
            missingness_df['column'] == col
        ]['missing_pct'].values
        if len(pct) > 0:
            print(f"   {col:<30} {pct[0]:>9.1f}%")
    if len(cols_high_missing) > 20:
        print(f"   ... and {len(cols_high_missing)-20} more")

print(f"\n {X_clean.shape[1]} columns remaining")